# Uncertain-input GP

連続入力の測定誤差とカテゴリ同定の不確かさを、output noise と区別して扱います。


## 連続入力の不確かさ


In [ ]:
import torch
from robotorchan.models import UncertainInputSingleTaskGP, UncertainCategoricalSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.linspace(0, 1, 10).unsqueeze(-1)
Y = torch.sin(2 * torch.pi * X)
X_std = torch.full_like(X, 0.03)
model = UncertainInputSingleTaskGP(X, Y, train_X_std=X_std)
print(model.posterior(torch.tensor([[0.25], [0.75]])).mean)


## カテゴリ確率を周辺化


In [ ]:
X_cont = torch.linspace(0, 1, 6).unsqueeze(-1)
P = torch.tensor([[0.9, 0.1], [0.8, 0.2], [0.6, 0.4], [0.4, 0.6], [0.2, 0.8], [0.1, 0.9]])
Y_cat = torch.sin(2 * torch.pi * X_cont) + 0.2 * P[:, 1:2]
cat_model = UncertainCategoricalSingleTaskGP(X_cont, P, Y_cat)
post = cat_model.posterior_with_category_probabilities(
    torch.tensor([[0.5]]), torch.tensor([[0.5, 0.5]])
)
print(post.mean)


## Uncertain-input candidate proposal

連続入力の不確実性を考慮したモデルから候補を選択します。カテゴリ確率モデルは別の予測APIを使うため、確率ベクトルを固定した候補比較を行います。


In [ ]:
candidate_grid = torch.linspace(0, 1, 51).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
assert candidate.shape == (1, 1)
print("uncertain-input candidate:", candidate)
cat_grid = torch.linspace(0, 1, 21).unsqueeze(-1)
cat_probs = torch.tensor([[0.5, 0.5]]).expand(len(cat_grid), -1)
cat_model.eval()
with torch.no_grad():
    cat_scores = cat_model.posterior_with_category_probabilities(
        cat_grid, cat_probs
    ).mean.squeeze(-1)
cat_candidate = cat_grid[cat_scores.argmax().item():cat_scores.argmax().item() + 1]
print("uncertain-category candidate:", cat_candidate)
